In [1]:
%pip install torch transformers pillow pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import pandas as pd
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# INITIALIZING THE PRE-TRAINED MODEL (CLIP)
print("Loading Pre-trained CLIP Model (This may take a moment to download on first run)...")
# Using OpenAI's CLIP model which can classify images based on custom text prompts
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)

# DEFINING THE FEATURE CATEGORIES (PROMPTS)
# Defining the categories we want the model to choose from for each feature
FEATURE_PROMPTS = {
    "Number_of_Stories": [
        "a photo of a one story house", 
        "a photo of a two story house", 
        "a photo of a three story house",
        "a photo of a four story house"
    ],
    "Roof_Type": [
        "a house with a concrete roof", 
        "a house with a tin or metal sheet roof", 
        "a house with a thatched, tarpaulin, or temporary roof"
    ],
    "Wall_Composition": [
        "a house with finished concrete or plastered brick walls", 
        "a house with exposed unplastered brick walls", 
        "a house with temporary mud, wood, or scrap tin walls"
    ],
    "Structural_Condition": [
        "a house in excellent, sturdy structural condition",
        "a house in average structural condition",
        "a house in poor, dilapidated structural condition with makeshift repairs"
    ]
}

def analyze_image_zero_shot(image_path):
    """Passes the image to CLIP and returns the best matching text for each feature."""
    try:
        image = Image.open(image_path).convert("RGB")
        extracted_features = {}
        
        # Analyzing the image for each feature category
        for feature_name, prompts in FEATURE_PROMPTS.items():
            # Preparing the inputs for the model
            inputs = processor(text=prompts, images=image, return_tensors="pt", padding=True)
            
            # Getting the model's predictions
            with torch.no_grad():
                outputs = model(**inputs)
            
            # The logits indicate how well the image matches the text prompts
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1) 
            
            # Finding the index of the highest probability
            best_match_idx = probs.argmax().item()
            
            # Cleaning up the prompt text to use as a label (e.g., removing "a photo of a ")
            best_label = prompts[best_match_idx].replace("a photo of a ", "").replace("a house with ", "").replace("a house in ", "")
            
            # For Structural Condition, let the model compute a continuous score based on its confidence distribution across the three categories.
            if feature_name == "Structural_Condition":
                prob_excellent = probs[0][0].item()
                prob_average = probs[0][1].item()
                prob_poor = probs[0][2].item()
                
                # Weighted sum: Excellent (1.0), Average (0.5), Poor (0.0)
                # This gives a dynamic score from 0.0 to 1.0 decided entirely by the model
                dynamic_score = (prob_excellent * 1.0) + (prob_average * 0.5) + (prob_poor * 0.0)
                extracted_features['Structural_Score'] = round(dynamic_score, 3)
                
            extracted_features[feature_name] = best_label.title()
            
        return extracted_features
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

# MAIN PIPELINE
def extract_and_merge_features(aggregated_excel_path, image_folder, output_excel_path):
    print(f"Loading aggregated dataset: {aggregated_excel_path}")
    df = pd.read_excel(aggregated_excel_path)
    
    vision_data = []
    
    print("\nScanning images and extracting visual features...")
    for index, row in df.iterrows():
        fid = row['FID']
        
        # Looking specifically for .jpg extension in the IMAGES folder
        image_path = os.path.join(image_folder, f"{fid}.jpg")
        
        if os.path.exists(image_path):
            print(f"[{index+1}/{len(df)}] Processing image for FID {fid}...")
            
            features = analyze_image_zero_shot(image_path)
            
            if features:
                vision_data.append({
                    'FID': fid,
                    'Image_Found': 'Yes',
                    'Stories': features['Number_of_Stories'],
                    'Roof_Type': features['Roof_Type'],
                    'Wall_Type': features['Wall_Composition'],
                    'Structural_Condition': features['Structural_Condition'],
                    'Structural_Score': features['Structural_Score']
                })
            else:
                # Image found but corrupted/unreadable
                vision_data.append({'FID': fid, 'Image_Found': 'Corrupted'})
        else:
            # No image found in the folder for this FID
            print(f"[{index+1}/{len(df)}] No image found for FID {fid}. Skipping vision extraction.")
            vision_data.append({'FID': fid, 'Image_Found': 'No'})

    # Converting the extracted features list into a DataFrame
    vision_df = pd.DataFrame(vision_data)
    
    # Merging the new visual features with the original dataset
    print("\nMerging visual features into the main dataset...")
    final_df = pd.merge(df, vision_df, on='FID', how='left')
    
    # Saving the updated dataset
    final_df.to_excel(output_excel_path, index=False, engine='openpyxl')
    print(f"\n--- Success! Dataset saved to {output_excel_path} ---")
    
    return final_df

if __name__ == "__main__":
    # Defining paths here
    INPUT_DATASET = "Aggregated_BPL_Families.xlsx" # The file we generated in the previous step
    IMAGE_DIRECTORY = "./IMAGES/"                # The folder where your .jpg images are stored
    OUTPUT_DATASET = "Final_BPL_Features.xlsx"     # The final dataset name
    
    # Creating mock directory if it doesn't exist so the script doesn't crash on test run
    os.makedirs(IMAGE_DIRECTORY, exist_ok=True)
    
    # Checking if the input file exists before running
    if os.path.exists(INPUT_DATASET):
        extract_and_merge_features(INPUT_DATASET, IMAGE_DIRECTORY, OUTPUT_DATASET)
    else:
        print(f"Error: Could not find '{INPUT_DATASET}'. Please run the aggregation script from the previous step first.")

Loading Pre-trained CLIP Model (This may take a moment to download on first run)...


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

C:\Users\Harsh Datt\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Harsh Datt\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Loading aggregated dataset: Aggregated_BPL_Families.xlsx

Scanning images and extracting visual features...
[1/26] Processing image for FID 1BTJ0915...
[2/26] Processing image for FID 1FQN9637...
[3/26] Processing image for FID 1GBA1267...
[4/26] Processing image for FID 1SGR5381...
[5/26] Processing image for FID 2ECX4300...
[6/26] Processing image for FID 3CGS3602...
[7/26] Processing image for FID 3DOY3691...
[8/26] Processing image for FID 3GZT5339...
[9/26] Processing image for FID 3INZ1220...
[10/26] Processing image for FID 3VCR3405...
[11/26] Processing image for FID 4XRM1260...
[12/26] Processing image for FID 4YIX7374...
[13/26] Processing image for FID 5LTS7349...
[14/26] Processing image for FID 5NCD2023...
[15/26] Processing image for FID 5OUS9140...
[16/26] Processing image for FID 6BUT4662...
[17/26] Processing image for FID 6CFG4258...
[18/26] Processing image for FID 6SES1288...
[19/26] Processing image for FID 7KBO2244...
[20/26] Processing image for FID 7SBB1254...
[